> **구분: 수업 핵심**

# 01. VLM의 입력과 출력 — 60분

이 노트북은 이미지가 API 요청에 들어가는 전체 경로를 확인합니다. **개념 15분 → 이미지 바이트 10분 → message TODO 15분 → 실제 질의 15분 → 정리 5분** 으로 진행합니다.

오늘 기억할 한 문장은 `VLM = 이미지 정보 + 언어 질문 → 언어 답변`입니다. 필요한 실행 절차는 이 노트북에 있으며, [README](README.md)와 [용어집](GLOSSARY.md)은 추가 참고 자료입니다.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

이미지를 보는 모델도 파일 경로 문자열만 주면 실제 사진을 읽는 것은 아닙니다. 이미지 파일의 bytes를 읽어 전송 형식으로 넣고 질문과 함께 전달해야 합니다. 이 실습은 영상 입력이 언어 모델 요청에 합쳐지는 경로를 이해하는 출발점입니다.

### 전체 과정에서 내 코드가 하는 일

```text
실제 이미지 파일
  ↓
bytes/Base64
  ↓
images가 있는 메시지 TODO
  ↓
vision 모델 요청
  ↓
원본·응답·조건 저장
```

빈 입력과 메시지 구조를 검사합니다. Base64는 전송 표기이며 영상 특징이나 의미를 추출한 결과가 아닙니다. 모델 이름만 보고 이미지 입력 지원을 가정하지 않습니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** Base64 문자열이 길면 이미지 이해가 더 정확한가요?

**설명:** 아닙니다. 전송 표기의 길이이며 인식 정확도와 같지 않습니다. 원본에 무엇이 실제로 보이는지와 응답의 근거를 나란히 보고 모델 capability·입력·질문 조건을 기록합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**2일차 VLM 연결:** 실제 질의에는 공용 Ollama가 필요합니다. 연결되지 않거나 모델이 없으면 로컬 Setup 담당자에게 서버·모델 저장소 확인을 요청합니다. 준비 여부는 **개인 컨테이너 JupyterLab Terminal** 에서 확인합니다.

```bash
cd /workspace
/opt/course/bin/python 05.VLM/scripts/check_environment.py
```

기본 API는 `http://127.0.0.1:11434`, 모델은 `gemma4:e2b-it-qat`이며 `vision` capability가 필요합니다. 주소·모델을 따로 배정받았다면 Python 초기화 셀에서 `os.environ['OLLAMA_HOST']`·`os.environ['VLM_MODEL']`을 설정합니다. 다른 Terminal에서 설정한 환경변수는 이미 열린 커널에 자동 적용되지 않습니다. 입력은 `Data/images/05.VLM/{road.png,dog.jpg}`, 선택 영상은 `Data/videos/05.VLM/ETS.mp4`, 출력은 `Data/outputs/05.VLM`입니다. 파일·모델이 없으면 배포 자산 경로를 확인합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[04.LLM/06_ollama_python.ipynb](../04.LLM/06_ollama_python.ipynb)의 요청 구조에서 필요한 내용은 `messages=[{'role':'user','content':'질문'}]`, `stream=False`로 완성된 응답을 받고 `response['message']['content']`에서 답을 읽는 것입니다. 여기서는 user message에 `images=[Base64 문자열]`을 더합니다. HTTP 호출은 제공된 `OllamaVLMClient`가 맡으므로 이전 LLM TODO를 완료하지 않아도 이 노트북의 독립 초기화부터 시작할 수 있습니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–5절의 수업 핵심 경로** → [2일차 슬라이드 52쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s047) · [배포 PDF 52쪽](../슬라이드/2일차.pdf#page=52)에서 59쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 같은 이미지의 요청·원문·검증 결과·지연을 저장합니다.
- **단원 마지막 연결 질문:** 공통 사례의 기대 label을 응답을 보기 전에 정합니다. 형식 통과와 업무 판단 정답을 따로 기록하고, 틀린 허용·불필요한 거부·판단 보류를 구분합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


> **구분: 수업 핵심**

## 완료 조건과 실행 전 예측

- `road.png`를 화면에 표시하고 파일 byte 수와 Base64 길이를 설명합니다.
- `build_vision_message` TODO가 정상·빈 입력 검증을 통과합니다.
- 실제 `gemma4:e2b-it-qat`가 `vision` capability를 가진 것을 확인하고 이미지 질문을 한 번 보냅니다.

실행 전 예상해 보세요. 이미지 1MB를 Base64로 바꾸면 문자열은 1MB보다 작아질까요, 커질까요? Base64를 다시 decode하면 원본과 완전히 같아야 할까요?


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import json, subprocess, sys, time
from IPython.display import Image, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '05.VLM/common').is_dir())
COURSE = ROOT / '05.VLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import IMAGES, VIDEOS, OUTPUTS, LOGS, ensure_output_dirs
from common.assets import compact_image_info, encode_image
from common.client import OllamaVLMClient, timing_summary
from common.loading import load_op
ensure_output_dirs()
assert (IMAGES / 'road.png').is_file() and (IMAGES / 'dog.jpg').is_file(), '먼저 강사에게 Data/images/05.VLM 배포를 확인하세요.'
print('Python:', sys.executable)
print('input:', IMAGES)
print('output:', OUTPUTS)


> **구분: 수업 핵심**

## 1. LLM과 VLM은 입력 계약이 다릅니다

LLM 요청에는 질문 글만 있습니다. VLM 요청의 user message에는 `content`와 `images`가 함께 있습니다. `images`는 경로 문자열을 REST 서버가 대신 읽어 주는 칸이 아니라 **client가 읽어 Base64로 바꾼 이미지 데이터 목록** 입니다.

| 단계 | 입력 | 처리 | 출력 |
|---|---|---|---|
| 파일 읽기 | PNG/JPEG 경로 | 크기·헤더 검사 | `bytes` |
| 전송 준비 | 이미지 `bytes` | Base64 encode | ASCII `str` |
| API | prompt + Base64 | vision encoder + language model | 응답 token |

`role='user'`는 사용자가 제공한 입력이라는 의미입니다. `content`는 모델이 수행할 질문, `images`는 그 질문의 시각 근거입니다.


In [ ]:
# 구분: 수업 핵심 셀
road = IMAGES / 'road.png'
display(Image(filename=str(road), width=640))
info = compact_image_info(road)
encoded = encode_image(road)
print('원본 byte:', info['bytes'])
print('Base64 문자 수:', len(encoded))
print('약 몇 배인가:', round(len(encoded) / info['bytes'], 3))
print('앞 24자만 확인:', encoded[:24] + '...')
assert len(encoded) > info['bytes']
print('예측 확인: Base64는 전송 표기이며 압축이나 암호화가 아닙니다.')


> **구분: 수업 핵심**

## 2. TODO — 이미지 message 만들기

파일 편집기에서 [exercises/message.py](exercises/message.py)를 엽니다. `build_vision_message(prompt, image_base64)`가 다음 dict를 반환하게 구현하세요.

```python
{'role': 'user', 'content': '정리된 질문', 'images': ['Base64 문자열']}
```

`prompt`는 앞뒤 공백을 제거합니다. prompt나 Base64가 문자열이 아니거나 비어 있으면 `ValueError`를 발생시킵니다. 빈 입력을 서버까지 보내 늦게 실패시키지 않고 입력 경계에서 바로 알려 주기 위한 규칙입니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `prompt='  무엇이 보이나요?  '`, `image_base64='YWJj'`이면 `{'role':'user','content':'무엇이 보이나요?','images':['YWJj']}`입니다. `'YWJj'`는 구조 확인용 예시이며 실제 이미지 요청에 쓰는 이미지 데이터는 아닙니다.
**작성 흐름 힌트:** prompt의 타입·strip 결과 검사 → Base64 입력이 비어 있지 않은 문자열인지 검사 → prompt만 strip → images에 문자열을 **리스트로 감싸** dict 반환. 파일 읽기·Base64 변환·서버 호출은 앞뒤 제공 코드의 역할입니다.
**설명할 지점:** “images에 파일 경로가 들어가는 것이 아니라 변환된 문자열이 들어갑니다. 이 TODO는 Base64 디코딩이나 이미지 유효성 검사를 구현하는 문제가 아닙니다.”
**수정 범위·통과 기준:** `build_vision_message`의 함수 본문만 완성하고, 실제 이미지 인코딩·API 호출은 제공 셀에 맡깁니다. 이미지 문자열은 공백 제거 없이 그대로 목록에 넣습니다. `check_message('exercise')`에서 정상 dict·질문 공백 정리·빈 값 거부를 확인하고 `message TODO PASS`를 봅니다. 에러가 나면 파일 저장 후 이 검증 셀부터 다시 실행합니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
from tests.test_course import check_message
check_message('exercise')
print('message TODO PASS')


In [ ]:
# 구분: 수업 핵심 셀
# 참고 정답은 강사가 안내하는 시점에 파일 편집기에서 확인합니다.
solution_path = COURSE / 'solutions/message.py'
print('참고 정답 경로:', solution_path)
# 수강생 파일을 자동으로 덮어쓰지 않습니다.


### 정답 해설 — 입력부터 결과까지

**전송 구조:** 정답은 빈 질문·빈 이미지 문자열을 먼저 거부하고, `role='user'`, `content=prompt.strip()`, `images=[image_base64]`를 만듭니다. images는 한 장이어도 목록입니다. base64는 이미지 bytes를 전송할 문자열로 표현한 것이며 이미지 특징을 추출한 값이 아닙니다.

질문 `'  무엇이 보이나요?  '`는 앞뒤 공백을 제거해 전송합니다. 이 함수는 문자열이 실제로 올바른 이미지인지 또는 모델이 사진을 정확하게 읽는지까지 검증하지 않습니다. 메시지 구조 검사→실제 요청→원본 사진과 응답 대조를 따로 수행합니다.

**확인 질문:** images에 파일 경로만 넣는 것과 이미지 내용을 인코딩해 넣는 것은 왜 다른가요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/message.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""01_vlm_basics.ipynb 참고 정답."""


def build_vision_message(prompt, image_base64):
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt는 비어 있지 않은 문자열이어야 합니다.")
    if not isinstance(image_base64, str) or not image_base64:
        raise ValueError("image_base64는 비어 있지 않은 문자열이어야 합니다.")
    # 질문만 정리하고 이미지는 API가 요구하는 list 안에 넣습니다.
    # Base64 생성·실제 이미지 검사는 이 함수 밖에서 수행합니다.
    return {"role": "user", "content": prompt.strip(), "images": [image_base64]}
```

</details>
<!-- todo-review:answer:end -->


> **구분: 수업 핵심**

## 3. 실제 요청을 보내기 전 payload 읽기

<!-- procedure:lesson-17-e68de9:start -->
![이미지와 질문을 한 요청으로](../교재그림/절차/lesson-17.svg)

Base64는 전송 표기입니다. 이미지 의미를 추론하는 단계는 VLM 내부이며 응답은 원본과 대조합니다.
<!-- procedure:lesson-17-e68de9:end -->


Base64 전체를 출력하면 Notebook이 매우 커집니다. 수업에서는 key와 길이만 확인합니다. 아래 payload의 `stream=False`는 응답 조각을 여러 번 받지 않고 완성된 JSON 하나를 기다린다는 뜻입니다. `num_predict`는 생성할 출력 token 상한이며 글자 수와 같지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
op = load_op('message')
message = op.build_vision_message('이미지에서 직접 보이는 교통수단 두 종류를 한국어로 답하세요.', encoded)
payload_preview = {
    'model': 'gemma4:e2b-it-qat',
    'messages': [{**message, 'images': [f'<Base64 {len(encoded)} chars>']}],
    'stream': False,
    'think': False,
    'options': {'temperature': 0, 'seed': 2026, 'num_predict': 80},
}
print(json.dumps(payload_preview, ensure_ascii=False, indent=2))


> **구분: 수업 핵심**

## 4. 실제 `gemma4:e2b-it-qat` 이미지 질의

첫 호출은 약 7GB 모델을 메모리에 올리는 시간이 더해질 수 있습니다. 아래 셀은 모델 목록과 이름만 보지 않고 `/api/show`의 `vision` capability를 확인합니다. 답이 문법적으로 자연스러운지와 별도로 실제 이미지에서 교통수단을 찾을 수 있는지 사람이 확인하세요.


In [ ]:
# 구분: 수업 핵심 셀
client = OllamaVLMClient(timeout=300)
model_info = client.require_vision()
print('model:', client.model, 'capabilities:', model_info.get('capabilities'))
result = client.chat([message], num_predict=80)
print('\n응답:\n', result['message']['content'])
print('\n시간:\n', json.dumps(timing_summary(result), ensure_ascii=False, indent=2))
assert result['message']['content'].strip(), '응답이 비었습니다.'


### 실제 질의 결과 보관

아래 셀은 방금 받은 응답을 다시 요청하지 않고 입력 경로·SHA-256, 모델·질문·옵션, 응답 원문·지연을 새 실행 폴더에 저장합니다. Base64 전체는 저장·출력하지 않습니다. 저장 경로를 확인한 뒤 이미지와 원문을 나란히 평가합니다.

In [ ]:
# 구분: 수업 핵심 셀 — 방금 실행한 실제 응답을 보존
from common.recording import create_run_dir
query_record = {
    'input': compact_image_info(road),
    'model': client.model,
    'prompt': message['content'],
    'options': {'temperature': 0, 'seed': 2026, 'num_ctx': 4096, 'num_predict': 80},
    'think': False,
    'raw_response': result['message']['content'],
    'timing': timing_summary(result),
}
query_destination = create_run_dir(OUTPUTS, 'basics') / 'image-query.json'
query_destination.write_text(json.dumps(query_record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert query_destination.is_file()
print('saved:', query_destination)

> **구분: 수업 핵심**

## 5. 결과를 평가하는 법

자연스러운 문장은 자동으로 사실이 되지 않습니다. 답에 나온 객체마다 이미지 안에서 직접 위치를 가리켜 보세요. 보이지 않는 교통 신호나 사람의 의도를 모델이 덧붙였다면 환각입니다. 이번 검증은 정답 문장 전체를 비교하지 않고 `응답이 존재하는가`, `질문 형식을 따르는가`, `이미지 근거가 있는가`를 나누어 봅니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 이미지 message 검증·실제 질의·결과 저장

[2일차 슬라이드 52쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s047) · [배포 PDF 52쪽](../슬라이드/2일차.pdf#page=52)로 돌아가 **52–59쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습**

## 자습: 헤더·해시·무결성

PNG의 앞 8 byte와 JPEG의 앞 3 byte는 파일 형식을 빠르게 확인하는 signature입니다. SHA-256은 같은 byte인지 확인하는 식별값입니다. 이미지 의미를 판정하는 값은 아닙니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
for name in ('road.png', 'dog.jpg'):
    path = IMAGES / name
    data = path.read_bytes()
    print(name, 'header=', data[:8], 'sha256=', compact_image_info(path)['sha256'])


> **구분: 수업 핵심**

## 정리와 오류 해결

- `FileNotFoundError`: `/workspace/Data/images/05.VLM` 배포와 현재 경로를 확인합니다.
- `확장자와 실제 헤더`: 이름만 `.png`로 바꾼 파일인지 확인합니다.
- `vision capability가 없습니다`: `gemma3:1b` 같은 text-only 태그인지 확인합니다.
- `Connection refused`: Ollama 서버가 실행 중인지 확인하되 임의로 중복 서버를 띄우지 않습니다.

다음 노트북에서는 같은 이미지를 질문 방식만 바꾸어 비교합니다.
